# 전체 직업 업무 설명 임베딩·검색
기존 Job의 비어 있지 않은 work 전체를 처리합니다. 현재 데이터의 예상 대상은 552개 직업입니다.

- text-embedding-3-small / 768차원 / job-work-v1
- Job.work_embedding / job_work_embedding_768
- 03과 같은 입력 형식·해시로 기존 업무 벡터를 재사용합니다. 100개가 정상 저장되어 있으면 신규 대상은 452개입니다.
- PDF와 이전 Chunk 임베딩은 사용하지 않습니다. 삭제·초기화 셀은 없습니다.
- 셀 1~4는 조회·검사, 셀 5는 유료 임베딩 API 호출과 DB 기록, 셀 6은 인덱스 생성 및 검증입니다.
- 셀 설명 순서대로 실행하세요. 03의 삭제 셀을 다시 실행하면 기존 데이터가 지워집니다.



## 셀 1 — 독립적인 환경 준비와 DB 연결
04 커널에서 .env와 연결 함수를 새로 준비합니다. 비밀값은 출력하지 않습니다.


In [ ]:
import os
import json
import hashlib
import math
import time
from pathlib import Path
from dotenv import dotenv_values
from neo4j import GraphDatabase

PROJECT_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
    if (p / 'data/expanded_20261108/jobs.csv').is_file()
    and (p / '.env').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('팀 프로젝트 폴더 또는 하위 폴더에서 노트북을 실행하세요.')
env = dotenv_values(PROJECT_ROOT / '.env')
username = env.get('NEO4J_USERNAME') or env.get('NEO4J_USER')
if not username or any(not env.get(k) for k in
    ('NEO4J_URI', 'NEO4J_PASSWORD', 'NEO4J_DATABASE', 'OPENAI_API_KEY')):
    raise ValueError('.env의 DB 설정과 OPENAI_API_KEY를 확인하세요.')
MODEL = 'text-embedding-3-small'
DIM = 768
INPUT_VERSION = 'job-work-v1'
INDEX = 'job_work_embedding_768'
DATABASE = env['NEO4J_DATABASE']
driver = GraphDatabase.driver(env['NEO4J_URI'],
    auth=(username, env['NEO4J_PASSWORD']), connection_timeout=10,
    max_transaction_retry_time=15)

def run_cypher(query, **params):
    with driver.session(database=DATABASE) as session:
        return [r.data() for r in session.run(query, **params)]

print('DB 연결:', run_cypher('RETURN 1 AS ok'))
print('프로젝트:', PROJECT_ROOT)


## 셀 2 — 전체 직업·업무 원문·인덱스 검사
안정적인 Job.id와 source_id를 확인하고 비어 있는 work는 제외합니다. 충돌하는 인덱스는 자동 변경하지 않습니다.


In [ ]:
job_rows = run_cypher("""
MATCH (j:Job)
RETURN j.id AS id, j.source_id AS job_code, j.name AS job_name, j.work AS work
ORDER BY id
""")
if not job_rows:
    raise ValueError('Job 노드가 없습니다. 대상 DB를 확인하세요.')
ids = [r['id'] for r in job_rows]
codes = [str(r['job_code']) if r['job_code'] is not None else None for r in job_rows]
if any(not x for x in ids) or len(set(ids)) != len(ids):
    raise ValueError('Job.id 누락 또는 중복')
if any(not x for x in codes) or len(set(codes)) != len(codes):
    raise ValueError('Job.source_id 누락 또는 중복')
eligible = []
for row in job_rows:
    row['job_code'] = str(row['job_code'])
    if row['id'] != 'job:' + row['job_code']:
        raise ValueError(f"ID 불일치: {row['id']}")
    if not isinstance(row['job_name'], str) or not row['job_name'].strip():
        raise ValueError(f"직업명 누락: {row['id']}")
    if row['work'] is not None and not isinstance(row['work'], str):
        raise ValueError(f"work가 문자열이 아님: {row['id']}")
    if row['work'] and row['work'].strip():
        eligible.append(row)
print('Job:', len(job_rows), '/ 업무 설명 있음:', len(eligible),
      '/ 빈 업무 설명:', len(job_rows) - len(eligible))

def vector_indexes():
    return run_cypher("""SHOW VECTOR INDEXES
    YIELD name, state, labelsOrTypes, properties, options
    RETURN name, state, labelsOrTypes, properties, options""")

def check_index(index):
    config = index['options'].get('indexConfig', {})
    if (index['labelsOrTypes'] != ['Job']
        or index['properties'] != ['work_embedding']
        or config.get('vector.dimensions') != DIM
        or config.get('vector.similarity_function', '').lower() != 'cosine'):
        raise ValueError('기존 업무 벡터 인덱스 설정 충돌: ' + index['name'])

for index in vector_indexes():
    if index['name'] == INDEX:
        check_index(index)
    elif index['labelsOrTypes'] == ['Job'] and index['properties'] == ['work_embedding']:
        raise ValueError('같은 속성의 다른 인덱스가 있습니다: ' + index['name'])
print('기존 인덱스 검사 완료')


## 셀 3 — 전체 업무 설명 입력·해시·토큰 확인
03과 입력 형식이 정확히 같아야 기존 벡터를 재사용합니다. 원문은 자르지 않으며 길이 초과 항목이 있으면 중단합니다.


In [ ]:
import tiktoken
if not eligible:
    raise ValueError('업무 설명이 있는 직업이 없습니다.')
targets = [dict(r) for r in sorted(eligible, key=lambda r: r['id'])]
encoding = tiktoken.encoding_for_model(MODEL)
for row in targets:
    row['input_text'] = f"직업명: {row['job_name']}\n하는 일: {row['work']}"
    row['input_hash'] = hashlib.sha256(row['input_text'].encode('utf-8')).hexdigest()
    row['tokens'] = len(encoding.encode(row['input_text']))
    if row['tokens'] > 8191:
        raise ValueError(f"입력 한도 초과: {row['id']}, {row['tokens']} 토큰")
target_ids = [r['id'] for r in targets]
print('전체 대상 직업:', len(targets), '/ 예상 입력 토큰:', sum(r['tokens'] for r in targets))
print('최대 입력 토큰:', max(r['tokens'] for r in targets))
print('입력 예시:\n', targets[0]['input_text'])


## 셀 4 — 재사용 검사와 신규 호출량 확인
기존 벡터의 모델·기록 차원·실제 차원·입력 버전·해시를 확인합니다. 불일치 또는 본문 변경 시 덮어쓰지 않고 중단합니다.


In [ ]:
def load_states():
    rows = run_cypher("""
    MATCH (j:Job) WHERE j.id IN $ids
    RETURN j.id AS id, j.name AS name, j.work AS work,
           j.work_embedding IS NOT NULL AS ready,
           size(j.work_embedding) AS actual_dimension,
           j.work_embedding_model AS model,
           j.work_embedding_dim AS dimension,
           j.work_embedding_version AS version,
           j.work_embedding_hash AS input_hash
    """, ids=target_ids)
    if len(rows) != len(targets) or len({r['id'] for r in rows}) != len(targets):
        raise ValueError('표본 Job 누락 또는 중복')
    return {r['id']: r for r in rows}

def reusable(row, state):
    if state['name'] != row['job_name'] or state['work'] != row['work']:
        raise ValueError('표본 선택 후 본문 변경: ' + row['id'])
    if not state['ready']:
        return False
    if (state['model'] != MODEL or state['dimension'] != DIM
        or state['actual_dimension'] != DIM or state['version'] != INPUT_VERSION
        or state['input_hash'] != row['input_hash']):
        raise ValueError('기존 업무 벡터의 설정·출처 충돌: ' + row['id'])
    return True

states = load_states()
reuse = sum(reusable(r, states[r['id']]) for r in targets)
print('기존 업무 벡터 재사용:', reuse, '/ 새로 임베딩:', len(targets) - reuse)

pending_before = [r for r in targets if not reusable(r, states[r['id']])]
print('신규 예상 입력 토큰:', sum(r['tokens'] for r in pending_before))


## 셀 5 — 누락된 업무 벡터만 배치 생성·저장
실행 때마다 DB를 다시 조회해 완료분을 건너뜁니다. 10개씩 저장하고 오류가 나면 실패 ID와 단계가 출력됩니다. 같은 커널에서는 API 응답을 메모리에 보관하여 DB 저장 실패 후 재사용합니다. API 성공 직후 커널이 종료되어 DB에 저장하지 못한 배치는 재호출될 수 있습니다.


In [ ]:
from openai import OpenAI
client = OpenAI(api_key=env['OPENAI_API_KEY'], timeout=60, max_retries=2)
if '_work_vector_cache' not in globals():
    _work_vector_cache = {}
if 'work_run_log' not in globals():
    work_run_log = []

def cache_key(row):
    return (MODEL, DIM, INPUT_VERSION, row['input_hash'])

def embed_and_store(rows):
    states = load_states()
    pending = [r for r in rows if not reusable(r, states[r['id']])]
    print('대상:', len(rows), '/ DB 재사용:', len(rows)-len(pending))
    for start in range(0, len(pending), 10):
        batch = pending[start:start+10]
        need_api = [r for r in batch if cache_key(r) not in _work_vector_cache]
        stage = 'embedding'
        try:
            if need_api:
                response = client.embeddings.create(model=MODEL, dimensions=DIM,
                    input=[r['input_text'] for r in need_api])
                data = sorted(response.data, key=lambda item: item.index)
                if [x.index for x in data] != list(range(len(need_api))):
                    raise ValueError('임베딩 응답 개수 또는 순서 이상')
                for row, item in zip(need_api, data):
                    vector = item.embedding
                    if len(vector) != DIM or not all(math.isfinite(x) for x in vector):
                        raise ValueError('벡터 차원 또는 수치 이상')
                    _work_vector_cache[cache_key(row)] = vector
                work_run_log.append({'stage':'api_success',
                    'ids':[r['id'] for r in need_api],
                    'tokens':response.usage.total_tokens})
            stage = 'database'
            payload = [{**r, 'vector':_work_vector_cache[cache_key(r)]} for r in batch]
            def store(tx):
                count = tx.run("""
                UNWIND $rows AS row
                MATCH (j:Job {id: row.id})
                WHERE j.name = row.job_name AND j.work = row.work
                  AND (j.work_embedding IS NULL OR
                    (j.work_embedding_model = $model
                     AND j.work_embedding_dim = $dim
                     AND j.work_embedding_version = $version
                     AND j.work_embedding_hash = row.input_hash))
                SET j.work_embedding = row.vector,
                    j.work_embedding_model = $model,
                    j.work_embedding_dim = $dim,
                    j.work_embedding_version = $version,
                    j.work_embedding_hash = row.input_hash,
                    j.work_embedding_updated_at = datetime()
                RETURN count(j) AS saved
                """, rows=payload, model=MODEL, dim=DIM, version=INPUT_VERSION).single()['saved']
                if count != len(batch):
                    raise RuntimeError('Job 누락 또는 동시 변경 감지: 배치 저장 취소')
                return count
            with driver.session(database=DATABASE) as session:
                saved = session.execute_write(store)
            work_run_log.append({'stage':'db_success','ids':[r['id'] for r in batch]})
            print('저장 완료:', saved, '건')
        except Exception as exc:
            work_run_log.append({'stage':stage+'_failed',
                'ids':[r['id'] for r in batch], 'error_type':type(exc).__name__})
            print('실패 단계:', stage, '/ 유형:', type(exc).__name__,
                  '/ ID:', [r['id'] for r in batch])
            raise


embed_and_store(targets)


## 셀 6 — 전체 검증 및 업무 인덱스 확인
대상 전체의 재사용 검사를 다시 통과해야 완료입니다. 기존 인덱스가 있다면 설정을 확인해 사용하며, 없다면 생성합니다. 노드·관계는 추가하거나 삭제하지 않습니다.


In [ ]:
states = load_states()
verified = sum(reusable(r, states[r['id']]) for r in targets)
if verified != len(targets):
    raise RuntimeError(f'임베딩 누락: {len(targets) - verified}개')
print('검증된 전체 업무 벡터:', verified)
print('빈 업무 설명 제외:', len(job_rows) - len(targets))
print('이 커널에서 기록한 배치 API 입력 토큰:', sum(r.get('tokens', 0) for r in work_run_log))
print('실패 이력:', [r for r in work_run_log if r['stage'].endswith('_failed')])
for index in vector_indexes():
    if index['name'] == INDEX:
        check_index(index)
    elif index['labelsOrTypes'] == ['Job'] and index['properties'] == ['work_embedding']:
        raise ValueError('같은 속성의 다른 인덱스: ' + index['name'])
run_cypher(f"""
CREATE VECTOR INDEX {INDEX} IF NOT EXISTS
FOR (j:Job) ON (j.work_embedding)
OPTIONS {{indexConfig: {{
 `vector.dimensions`: 768,
 `vector.similarity_function`: 'cosine'
}}}}
""")
for _ in range(20):
    index = next((r for r in vector_indexes() if r['name'] == INDEX), None)
    if index is None: raise RuntimeError('인덱스 생성 확인 실패')
    check_index(index)
    if index['state'] == 'ONLINE': break
    if index['state'] == 'FAILED': raise RuntimeError('인덱스 생성 실패')
    time.sleep(2)
else:
    raise TimeoutError('아직 준비 중입니다. 잠시 후 이 셀을 다시 실행하세요.')
print(INDEX, index['state'])
print(run_cypher("""MATCH (j:Job)
RETURN count(j) AS jobs, count(j.work_embedding) AS embedded_jobs"""))


## 셀 7 — 전체 직업 대상 의미 검색
질문도 같은 small·768차원으로 임베딩합니다. 반환은 job_code·job_name·work 세 필드입니다. 생성 답변이 아닌 검색된 업무 원문을 확인하세요.


In [ ]:
def search_work(question, top_k=5):
    if not isinstance(question, str) or not question.strip():
        raise ValueError('질문을 입력하세요.')
    if not isinstance(top_k, int) or not 1 <= top_k <= 20:
        raise ValueError('top_k는 1~20 정수로 지정하세요.')
    if len(encoding.encode(question)) > 8191:
        raise ValueError('질문 입력 한도 초과')
    incompatible = run_cypher("""
    MATCH (j:Job) WHERE j.work_embedding IS NOT NULL
    AND (coalesce(j.work_embedding_model, '') <> $model
      OR coalesce(j.work_embedding_dim, -1) <> $dim
      OR size(j.work_embedding) <> $dim
      OR coalesce(j.work_embedding_version, '') <> $version)
    RETURN count(j) AS n
    """, model=MODEL, dim=DIM, version=INPUT_VERSION)[0]['n']
    if incompatible:
        raise ValueError('업무 인덱스에 다른 설정의 벡터가 있습니다.')
    current = load_states()
    if not all(reusable(r, current[r['id']]) for r in targets):
        raise ValueError('업무 벡터 누락: 셀 5~6을 확인하세요.')
    response = client.embeddings.create(model=MODEL, dimensions=DIM, input=[question])
    vector = response.data[0].embedding
    if len(vector) != DIM: raise ValueError('질문 벡터 차원 불일치')
    return run_cypher("""
    CALL db.index.vector.queryNodes($index, $k, $vector)
    YIELD node AS j, score
    RETURN toString(j.source_id) AS job_code,
           j.name AS job_name, j.work AS work
    ORDER BY score DESC
    """, index=INDEX, k=top_k, vector=vector)

for question in ['일기예보관은 어떤 일을 하나요?',
                 '날씨 관측 자료를 분석해 예보하는 직업은?']:
    print('\n질문:', question)
    results = search_work(question, top_k=3)
    assert all(set(r) == {'job_code','job_name','work'} for r in results)
    print(json.dumps(results, ensure_ascii=False, indent=2))


## 셀 8 — 검색 직업과 학과·자격·능력 관계 연결
job_code를 기존 Job.id에 연결합니다. 기본 그래프 재적재만 했다면 정밀 출처 속성이 null일 수 있습니다. 관련 자격은 필수 자격을 의미하지 않습니다.


In [ ]:
results = search_work('날씨 관측 자료를 분석해 예보하는 직업은?', top_k=1)
if results:
    job_id = 'job:' + results[0]['job_code']
    paths = run_cypher("""
    MATCH (j:Job {id: $id})-[r]->(target)
    WHERE type(r) IN ['RELATED_MAJOR','RELATED_CERTIFICATE','HAS_CORE_ABILITY']
    RETURN j.id AS job_id, type(r) AS relation, target.name AS target_name,
           r.source_file AS source_file, r.source_locator AS source_locator,
           r.source_record_json AS source_record_json
    ORDER BY relation, target_name
    """, id=job_id)
    print(json.dumps(paths, ensure_ascii=False, indent=2))
else:
    print('검색 결과가 없습니다.')


## 완료 기준
전체 업무 설명이 있는 Job의 수와 검증된 업무 벡터 수가 같고, 인덱스가 ONLINE이어야 합니다. 현재 데이터 기준 예상 수는 552개입니다. 검색 결과를 원문과 비교한 뒤 Supervisor 연동에 사용하세요.

이 파일은 작성 시점에 DB 쓰기·API 호출을 실행하지 않았습니다. 노트북을 저장하면 실행 결과가 함께 보존됩니다. 연결을 닫으려면 작업 종료 후 별도 셀에서 driver.close()를 실행하세요.

